In [1]:
# All paths in this notebook are relative to the repository root; anchor the working directory there
import os
while not os.path.exists('METHODOLOGY.md') and os.getcwd() != '/': os.chdir('..')
assert os.path.exists('METHODOLOGY.md'), 'run from inside the Nofit_LRT_Extension repository'

# LRT Capture — Designed Uncertainty Experiment (Step 40, task C8)

Steps 31/32 (`Mode_skims_and_flow_comparison.ipynb`) already run in about a minute and, per
combination of `GC_SOURCE_DIR` / `LRT_HEADWAY` / `BUS_COMPETITION`, already sweep five λ/premium
cases (task C4's `LAM_CASES`) across every LRT regime present in that GC source. This notebook
does not re-implement that sweep; it assembles the runs already made across tasks C4–C7 (plus
two combinations run once, one-off, for this notebook — see §1) into a single factorial table
and a tornado figure, and records which of the plan's eight factors could and could not be
varied given this session's constraints.

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BLUE, ORANGE, AQUA, PURPLE, INK, INK2, MUTED, GRID, AXIS = '#2a78d6', '#eb6834', '#1baf7a', '#7b5bd6', '#0b0b0b', '#52514e', '#898781', '#e1e0d9', '#c3c2b7'
OUT = 'Output/skims/uncertainty'; os.makedirs(OUT, exist_ok=True); os.makedirs('Output/figures', exist_ok=True)

## 1. The runs assembled

`Mode_skims_and_flow_comparison.ipynb`'s own `OUT`-tagging (task C5/C7) only composes one
alternate dimension at a time: setting `GC_SOURCE_DIR` to an alternate step-26 directory makes
`OUT` follow it exactly, so a second switch (`BUS_COMPETITION`) applied on top would silently
overwrite that directory's existing `full`-competition results rather than tag alongside them.
Four of the six headway × bus-competition cells below were already on disk from tasks C4/C5/C7;
the two `truncated` cells at headway 7.5 / 10 were produced by one-off runs of the unmodified
notebook (`LRT_HEADWAY` and `BUS_COMPETITION` both set, `GC_SOURCE_DIR` pointed at that
headway's step-26 directory), with only `lrt_capture_scenarios.csv` copied out to a new,
non-colliding directory (`Output/skims/lrt_headway_{7.5,10}_truncated/`) — the shared
`Output/skims/lrt_headway_{7.5,10}/` directories and the notebook itself were left untouched
(`git checkout` after each run). This is a data-assembly convenience, not a code fix; the
`OUT`-tagging gap itself is left as found, since fixing it touches a shared default code path
for the sake of two cells this notebook already has by other means.

In [3]:
SOURCES = {
    (5.0, 'full'):      'Output/skims/lrt_capture_scenarios.csv',
    (7.5, 'full'):      'Output/skims/lrt_headway_7.5/lrt_capture_scenarios.csv',
    (10.0, 'full'):     'Output/skims/lrt_headway_10/lrt_capture_scenarios.csv',
    (5.0, 'truncated'): 'Output/skims/bus_truncated/lrt_capture_scenarios.csv',
    (7.5, 'truncated'): 'Output/skims/lrt_headway_7.5_truncated/lrt_capture_scenarios.csv',
    (10.0, 'truncated'):'Output/skims/lrt_headway_10_truncated/lrt_capture_scenarios.csv',
}
rows = []
for (hw, bc), f in SOURCES.items():
    d = pd.read_csv(f); d['headway_min'] = hw; d['bus_competition'] = bc; rows.append(d)
fac = pd.concat(rows, ignore_index=True)
fac.to_csv(f'{OUT}/lrt_capture_factorial.csv', index=False)
print(f"{len(fac)} rows: {fac['scenario'].nunique()} regimes x {fac.groupby('scenario').size().max()} (headway x bus_competition x lambda/premium) cells at most")
print(fac.groupby('scenario').size())

155 rows: 6 regimes x 30 (headway x bus_competition x lambda/premium) cells at most
scenario
LRT + synthetic branches (task C6)     5
LRT all ground                        30
LRT all underground                   30
LRT design 50 km/h                    30
LRT design 50 km/h + accel/braking    30
LRT mixed (Haifa core underground)    30
dtype: int64


**Factors varied:** LRT regime (5 of the 6 step-26 regimes — `LRT + synthetic branches` (task
C6) only exists at headway 5 / full competition, since its GC comes from step 26 directly
rather than through the feeder/gateway loop the other regimes share, and was not re-run at the
other five headway/competition cells for this notebook), headway (5 / 7.5 / 10 min), λ and the
LRT premium (five cases around the central point: λ 0.02/0.03/0.05 with the premium held at 5,
and premium 0/5/10 with λ held central — a one-factor-at-a-time design around the centre, not a
full λ × premium cross, matching how task C4 already built `LAM_CASES`), and bus competition
(full / truncated, task C7 — `truncated` is an explicit upper bound, not a plausible range).

**Factors fixed, not varied (see the reminder at the end of this notebook and of the
handover):**
- **Coverage threshold** (step 15's `binary_guard` vs `all_ravkav` variants) — varying this
  means re-running steps 15 through 32, not just re-reading step 26/31's saved skims; out of
  scope for a same-day factorial built from already-computed runs.
- **Walk access source** (straight-line vs OSM network distance, tasks C2/C3) — blocked: this
  environment's egress policy rejects `download.geofabrik.de`, `overpass-api.de` and
  `www.openstreetmap.org` (403/CONNECT), and per policy that block is reported, not retried.
- **Car GC source** (2022 survey vs a Google-uplifted car skim, tasks C1/E5) — blocked: Google's
  traffic-aware routing only predicts a future `departure_time`, so it cannot return the
  historical May–2026 conditions the plan calls for, now that period is in the past. The client
  is expected to supply a real model-network car skim later (see the closing reminder).

## 2. The tornado — how much each *available* factor moves the central case

Central point: `LRT all underground`, headway 5 min, full bus competition, central λ/premium —
4,254 LRT trips 06:00–09:00 (task C4's own central case). Each factor below is swept alone,
the other three held at that central point.

In [4]:
CENTRAL_TRIPS = fac.query("scenario=='LRT all underground' and headway_min==5.0 and bus_competition=='full' and case.str.startswith('central')")['LRT trips 06–09'].iloc[0]

def factor_range(mask, label_col):
    sub = fac[mask][[label_col, 'LRT trips 06–09']].drop_duplicates().sort_values('LRT trips 06–09')
    return sub

central_mask = (fac.headway_min == 5.0) & (fac.bus_competition == 'full') & (fac.case.str.startswith('central'))

tornado_rows = []
# regime (excluding the synthetic-branches special case, only run at the central point)
sub = factor_range(central_mask & (fac.scenario != 'LRT + synthetic branches (task C6)'), 'scenario')
tornado_rows.append({'factor': 'LRT regime', 'low_label': sub.iloc[0]['scenario'], 'low': sub.iloc[0]['LRT trips 06–09'],
                      'high_label': sub.iloc[-1]['scenario'], 'high': sub.iloc[-1]['LRT trips 06–09']})
# headway
sub = factor_range((fac.scenario == 'LRT all underground') & (fac.bus_competition == 'full') & (fac.case.str.startswith('central')), 'headway_min')
tornado_rows.append({'factor': 'LRT headway (min)', 'low_label': f"{sub.iloc[0]['headway_min']:g}", 'low': sub.iloc[0]['LRT trips 06–09'],
                      'high_label': f"{sub.iloc[-1]['headway_min']:g}", 'high': sub.iloc[-1]['LRT trips 06–09']})
# lambda / premium (5 cases at the true central regime/headway/competition)
sub = fac[(fac.scenario == 'LRT all underground') & (fac.headway_min == 5.0) & (fac.bus_competition == 'full')][['case', 'LRT trips 06–09']].sort_values('LRT trips 06–09')
tornado_rows.append({'factor': 'λ / LRT premium (5 cases)', 'low_label': sub.iloc[0]['case'], 'low': sub.iloc[0]['LRT trips 06–09'],
                      'high_label': sub.iloc[-1]['case'], 'high': sub.iloc[-1]['LRT trips 06–09']})
# bus competition
sub = factor_range((fac.scenario == 'LRT all underground') & (fac.headway_min == 5.0) & (fac.case.str.startswith('central')), 'bus_competition')
tornado_rows.append({'factor': 'Bus competition on trunk', 'low_label': sub.iloc[0]['bus_competition'], 'low': sub.iloc[0]['LRT trips 06–09'],
                      'high_label': sub.iloc[-1]['bus_competition'], 'high': sub.iloc[-1]['LRT trips 06–09']})

tor = pd.DataFrame(tornado_rows)
tor['range'] = tor['high'] - tor['low']
tor = tor.sort_values('range')
tor.to_csv(f'{OUT}/lrt_capture_tornado.csv', index=False)
print(tor.to_string(index=False))

                   factor                       low_label    low                     high_label   high  range
        LRT headway (min)                              10 3410.0                              5 4254.0  844.0
 Bus competition on trunk                            full 4254.0                      truncated 5754.0 1500.0
               LRT regime                  LRT all ground 3206.0             LRT design 50 km/h 5095.0 1889.0
λ / LRT premium (5 cases) high λ (0.05 / 0.10, premium 5) 3166.0 low λ (0.02 / 0.03, premium 5) 6031.0 2865.0


In [5]:
fig, ax = plt.subplots(figsize=(8, 4))
y = np.arange(len(tor))
ax.barh(y, tor['high'] - CENTRAL_TRIPS, left=CENTRAL_TRIPS, color=AQUA, label='high')
ax.barh(y, tor['low'] - CENTRAL_TRIPS, left=CENTRAL_TRIPS, color=ORANGE, label='low')
ax.axvline(CENTRAL_TRIPS, color=INK, lw=1)
ax.set_yticks(y); ax.set_yticklabels(tor['factor'])
ax.set_xlabel('LRT trips 06:00–09:00 (central case = ' + f'{CENTRAL_TRIPS:,.0f})')
ax.set_title('Task C8 — tornado of the factors this session could actually vary')
ax.legend(loc='lower right'); ax.grid(axis='x', color=GRID, lw=0.6)
fig.tight_layout(); fig.savefig('Output/figures/lrt_capture_tornado.png', dpi=150)
plt.show()

## Findings

Ranked by the size of the move on the central case (4,254 LRT trips), smallest to largest:
**headway** (5 -> 10 min: 4,254 -> 3,410, range 844), **bus competition on trunk** (full ->
truncated: 4,254 -> 5,754, range 1,500), **LRT regime** (all-ground -> design 50 km/h: 3,206 ->
5,095, range 1,889), and **lambda / the LRT premium** (high lambda -> low lambda: 3,166 -> 6,031,
range 2,865, the widest of the four). Bus competition's range is task C7's explicit upper bound
(today's parallel bus service removed as an alternative entirely), not a plausible band, so of the
three genuine-uncertainty factors, lambda/premium and regime move the case by a comparable, larger
amount than headway; none of the four is small enough to drop from a future full design.

**What this factorial is not:** three of the plan's eight factors (coverage threshold, walk access
source, car GC source) are held fixed at their only available value, not varied — this table is a
factorial over five of eight factors, not the full design C8 originally specified. Coverage needs a
rerun from step 15; walk access is blocked by this environment's egress policy on OSM sources (see
Section 1); car GC source is blocked by the same historical-traffic gap task E5/C1 already found
with Google's API, and is waiting on the client's own model-network car skim (see
`docs/NEXT_STEPS_HANDOVER_2026-09-23.md`, item C1, and the reminder closing that document).
